# PrismLab P4-D.1 — Statistical Validation Addendum

**Status:** Prospective validation protocol. **No statistics or performance measurements are invented in this notebook.**

This notebook supplements, rather than replaces, `PrismLab_P4_Research_Checkpoint.ipynb`. The original single-run P4-D result remains **8/10 for both architectures**. Generate validation results using the linked CLI scripts **before** interpreting comparisons here.

Two complementary stages:
- **Repeatability experiment:** existing 10-case dataset, e.g. five repeats per architecture; measures output stability, timing jitter and execution robustness **conditional on these ten previously viewed cases**.
- **Generalization experiment:** 40 new frozen image/audio combinations (24 answerable, 16 abstention), generated **before new inference**, reviewed manually, and evaluated on both pipelines. A single holdout pass estimates unseen-case performance; repetitions add runtime reproducibility.

**No claim of statistical significance is made automatically.**

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidate_roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidate_roots if (p/'data/p4/robustness/manifest.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open within the lfm3-prismlab project root/notebooks directory')
RESULTS = ROOT/'results/p4/stat_validation'
print('Project root:', ROOT)
print('Statistical validation reports:', RESULTS)
print('Existing files:', *sorted(p.name for p in RESULTS.glob('*.json')) if RESULTS.exists() else [], sep='\n - ')

## 1. Pre-inference audit

The new holdout must remain frozen and require human visual verification. The notebook does **not** alter a manifest, regenerate labels, or inspect a model while creating expected answers. Confirm by sight that missing relations truly have empty neighboring cells; verify all 40 labels and listen to the five reused recordings. Distinct visual scenes share spoken questions, so they are not 40 independent audio recordings.

In [ ]:
for rel in ['data/p4/robustness/manifest.json', 'data/p4/holdout_v1/manifest.json']:
    path = ROOT/rel
    if path.exists():
        d = json.loads(path.read_text())
        counts = pd.Series([c['expected']=='unknown' for c in d['cases']]).value_counts()
        print('\nDataset:', d['dataset'], '| cases:', len(d['cases']), '| status:', d.get('status'))
        print('Requires abstention:', int(counts.get(True, 0)), '| answerable:', int(counts.get(False, 0)))
    else:
        print('Not yet created:', rel)

## 2. Load paired trial files

Both architectures must use **the same manifest SHA256, case IDs, number of repetitions, and randomization seed**. Run `p4_validation_analyze.py` to compute the case-clustered intervals and save `analysis_*.json`/`.csv`. Then execute the cells below. Missing outputs are shown as *not yet run*, not as zero accuracy.

In [ ]:
if not RESULTS.exists():
    print('No validation files yet. Run p4_validation_runner.py on the laptop first.')
else:
    files = sorted(RESULTS.glob('analysis_*.json'))
    print('Available statistical analyses:', len(files))
    for f in files:
        d = json.loads(f.read_text())
        ci = d['accuracy_diff_95pct_case_bootstrap_CI']
        print(f.name, '| independent visual cases:', d['independent_cases'],
              '| repeats:', d['repeats_per_case'],
              '| native:', f"{d['native_strict_rate']:.1%}",
              '| modular:', f"{d['modular_strict_rate']:.1%}",
              '| case-bootstrap accuracy Δ CI:', tuple(round(x,3) for x in ci))

## 3. Paired results and visualizations

Case is the unit for a case-level confidence interval. Repeated same-case responses are **not independent case samples**. For the 40-case holdout, confidence intervals resampling the five repeated *question templates* as blocks offer a useful, generally wider sensitivity check.

In [ ]:
if RESULTS.exists():
    for f in sorted(RESULTS.glob('analysis_*.csv')):
        df = pd.read_csv(f)
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
        x = list(range(len(df)))
        axes[0].scatter(x, df['native_accuracy'], label='Native Omni', marker='o', s=22)
        axes[0].scatter(x, df['modular_accuracy'], label='Modular Audio → VL', marker='x', s=22)
        axes[0].set_ylim(-0.05, 1.05); axes[0].set_xlabel('Case index')
        axes[0].set_ylabel('Strict pass fraction')
        axes[0].legend(); axes[0].set_title('Within-case repeatability')
        axes[1].scatter(df['native_median_s'], df['modular_median_s'], s=25)
        maxval = max(df[['native_median_s','modular_median_s']].max())*1.05
        axes[1].plot([0,maxval],[0,maxval], linestyle='--')
        axes[1].set_xlabel('Omni median (s)'); axes[1].set_ylabel('Modular median (s)')
        axes[1].set_title('Matched case latencies')
        plt.suptitle(f.stem)
        plt.tight_layout(); plt.show()
        display(df.head(8))
else:
    print('No result files yet.')

## 4. Statistical interpretation — do not delete these qualifications

- **Repeatability:** Five or ten runs of the same image/audio pair measure run-to-run variation, not five or ten new examples.
- **Holdout:** A 40-case test built with new shapes and spatial arrangements is a much stronger generalization check than reusing D01–D10. Yet it still comes from one synthetic generator and five reused spoken questions.
- **Paired confidence interval:** Report 95% case-cluster bootstrap CI on (modular − native) accuracy. Inspect question-family block bootstrap as a dependency sensitivity check.
- **Significance:** Exact case-level McNemar (accuracy) and paired case-level sign test (latency) are exploratory; with a small number of independent case families, a p-value is not a universal architectural guarantee.
- **Latency confound:** The dual-resident modular run and single-resident Omni run differ in memory residency. CPU thermal effects, OS load, caching, and fixed phase order are additional limitations.
- **No leakage:** Do not tune prompts or thresholds after examining holdout outputs and then present scores as untouched holdout accuracy.

**Claim upgrade if new experiments succeed:** “In a preconstructed 40-case synthetic holdout with fixed 4-bit CPU-local inference, we observed [measured metrics], with [measured intervals]. Findings are limited to tested visual scenes, questions, and runtime conditions.”

In [ ]:
# Report a concise evidence-based statement only when validation summaries exist.
if RESULTS.exists():
    for f in sorted(RESULTS.glob('analysis_*.json')):
        s = json.loads(f.read_text())
        delta = s['paired_accuracy_difference_modular_minus_native']
        ci = s['accuracy_diff_95pct_case_bootstrap_CI']
        print('\n' + f.name)
        print(f"Cases={s['independent_cases']}, repeats={s['repeats_per_case']}")
        print(f"Accuracy difference (modular-native): {delta:+.1%}; case 95% CI [{ci[0]:+.1%}, {ci[1]:+.1%}]")
        print('Question-template block CI:', s['accuracy_diff_95pct_question_block_bootstrap_CI'])
        print(f"Exact case-level McNemar p={s['case_level_majority_mcnemar_exact_p']:.4g}")
        print(f"Paired latency sign-test p={s['paired_case_latency_sign_test_p']:.4g}")
else:
    print('No statistical validation analyses yet.')

## 5. Next decision

Complete the **repeatability** and **unseen holdout** stages before replacing the original exploratory conclusion. If both architectures still fail absent spatial relations in the holdout, label that as a recurring failure pattern across test sets; do not call it a proven shared architectural defect. Independent human recordings, diverse scenes and hardware-matched trials are later improvements.